# Day 28 — Prediction Forms, Model Comparison, and Batch Analysis

This notebook adds a separate prediction dashboard page while preserving your existing Day 27 files. It audits existing routes and model manifests, creates the page assets, runs Django checks, and generates a report.

**Important:** it does not invent prediction inputs or metric values. Prediction requests require the actual schema expected by your registered API. URL changes are provided as a patch for review rather than overwriting your existing `urls.py`.


In [26]:
from pathlib import Path
import os, re, json, shutil, datetime, subprocess, sys

CWD = Path.cwd().resolve()
ROOT = next((p for p in [CWD, *CWD.parents] if (p / "django_api" / "manage.py").exists()), None)
if ROOT is None:
    ROOT = next((p.parent for p in [CWD, *CWD.parents] if (p / "manage.py").exists() and (p / "ipl_api").exists()), None)
if ROOT is None:
    raise FileNotFoundError("Open this notebook from your repository root or notebooks folder.")
DJANGO_DIR = ROOT / "django_api" if (ROOT / "django_api" / "manage.py").exists() else ROOT
APP_DIR = DJANGO_DIR / "ipl_api"
DOCS = ROOT / "docs"
DOCS.mkdir(parents=True, exist_ok=True)
stamp = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
BACKUP = ROOT / "backups" / f"day28_{stamp}"
BACKUP.mkdir(parents=True, exist_ok=True)
print("Repository:", ROOT)
print("Django directory:", DJANGO_DIR)
print("App directory:", APP_DIR)
print("Backup directory:", BACKUP)


Repository: D:\Git\Personal\ipl-player-performance
Django directory: D:\Git\Personal\ipl-player-performance\django_api
App directory: D:\Git\Personal\ipl-player-performance\django_api\ipl_api
Backup directory: D:\Git\Personal\ipl-player-performance\backups\day28_20261024_020056


## 1. Audit existing routes and model files

Inspect the actual URL configuration and prediction modules before merging the new page route.


In [27]:
def read_text(path):
    try:
        return path.read_text(encoding="utf-8")
    except Exception:
        return ""

url_files = [p for p in [APP_DIR / "urls.py", DJANGO_DIR / "config" / "urls.py"] if p.exists()]
prediction_files = sorted(APP_DIR.glob("*predict*.py")) + sorted(APP_DIR.glob("*batch*.py"))
manifest_paths = [
    ROOT / "reports/day10/model_manifest.csv",
    ROOT / "reports/day12/model_artifacts_manifest.csv",
    ROOT / "reports/day23/model_manifest.csv",
    ROOT / "reports/day23/model_artifacts_manifest.csv",
]
print("URL configurations:")
for p in url_files:
    print("\n---", p.relative_to(ROOT) if p.is_relative_to(ROOT) else p, "---")
    print(read_text(p)[:5000])
print("\nPrediction/batch modules:")
for p in prediction_files:
    print("-", p.relative_to(ROOT), f"({p.stat().st_size} bytes)")
print("\nModel manifests:")
for p in manifest_paths:
    print("-", p.relative_to(ROOT), "FOUND" if p.exists() else "not found")


URL configurations:

--- django_api\ipl_api\urls.py ---
from django.urls import path
from .views import PlayerListAPIView, PlayerDetailAPIView
from .dashboard_views_day28 import dashboard_day28
from .eda_summary_day26 import eda_summary_day26
from .eda_views import EDABattingAPIView, EDABowlingAPIView, EDASeasonAPIView, EDAVenueAPIView, EDAOppositionAPIView, EDARecentFormAPIView
from .prediction_views import BattingPredictionAPIView, BowlingPredictionAPIView, UnifiedPredictionAPIView
from .batch_prediction_views import BatchPredictionAPIView
from .player_name_prediction_views_day28 import player_name_prediction_day28, player_name_options_day28, model_metrics_day28

from .dashboard_views_day26 import dashboard_day25
from .dl_database_prediction_views_day33 import predict_dl_from_database
urlpatterns = [
    path("dashboard/", dashboard_day25, name="ipl-dashboard"),
    path("dashboard-day28/", dashboard_day28, name="ipl-dashboard-day28"),
    path("players/", PlayerListAPIView.as_view()

## 2. Back up current dashboard and URL files

This is a safety step. The notebook creates new Day 28 assets and does not replace `views.py` or `urls.py`.


In [28]:
backup_targets = [
    APP_DIR / "urls.py",
    APP_DIR / "views.py",
    APP_DIR / "dashboard_views_day26.py",
    APP_DIR / "templates/ipl_api/dashboard_day25.html",
    APP_DIR / "static/ipl_api/js/dashboard_day25.js",
    APP_DIR / "static/ipl_api/css/dashboard_day25.css",
]
for source in backup_targets:
    if source.is_file():
        dest = BACKUP / source.relative_to(ROOT)
        dest.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(source, dest)
        print("Backed up:", source.relative_to(ROOT))
print("Backup complete.")


Backed up: django_api\ipl_api\urls.py
Backed up: django_api\ipl_api\views.py
Backed up: django_api\ipl_api\dashboard_views_day26.py
Backed up: django_api\ipl_api\templates\ipl_api\dashboard_day25.html
Backed up: django_api\ipl_api\static\ipl_api\js\dashboard_day25.js
Backed up: django_api\ipl_api\static\ipl_api\css\dashboard_day25.css
Backup complete.


## 3. Create the Day 28 dashboard page

This page includes single prediction, batch analysis, and model metrics panels. Use only the request schema defined by your real prediction API.


In [29]:
TEMPLATE_DIR = APP_DIR / "templates/ipl_api"
JS_DIR = APP_DIR / "static/ipl_api/js"
CSS_DIR = APP_DIR / "static/ipl_api/css"
for folder in [TEMPLATE_DIR, JS_DIR, CSS_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

template_path = TEMPLATE_DIR / "dashboard_day28.html"
js_path = JS_DIR / "dashboard_day28.js"
css_path = CSS_DIR / "dashboard_day28.css"

template = '''<!doctype html>
<html lang="en">
<head>
<meta charset="utf-8"><meta name="viewport" content="width=device-width, initial-scale=1">
<title>IPL Player Analytics — Predictions</title>
<link href="https://cdn.jsdelivr.net/npm/bootstrap@5.3.3/dist/css/bootstrap.min.css" rel="stylesheet">
<link rel="stylesheet" href="/static/ipl_api/css/dashboard_day28.css">
</head>
<body>
<nav class="navbar navbar-dark bg-dark px-3"><span class="navbar-brand">IPL Player Analytics</span><span class="text-light small">Predictions • Model Comparison • Batch Analysis</span></nav>
<main class="container-fluid py-4">
<div class="d-flex flex-wrap justify-content-between align-items-center gap-2 mb-3">
<div><h1 class="h3 mb-1">Prediction & Model Evaluation</h1><p class="text-secondary mb-0">Use registered APIs and saved evaluation metrics.</p></div>
<a class="btn btn-outline-secondary" href="/api/dashboard/">Back to Day 27 dashboard</a>
</div>
<div id="status" class="alert alert-info" role="status">Checking summary API…</div>
<div class="row g-3">
<div class="col-12 col-xl-6"><section class="card h-100"><div class="card-header fw-semibold">Single prediction</div><div class="card-body">
<label class="form-label" for="predictionEndpoint">Prediction endpoint</label>
<select id="predictionEndpoint" class="form-select mb-3"><option value="">Select only after confirming it is registered</option><option value="/api/predict/batting/">/api/predict/batting/</option><option value="/api/predict/bowling/">/api/predict/bowling/</option><option value="/api/predict/">/api/predict/</option></select>
<label class="form-label" for="predictionPayload">Request JSON payload</label><textarea id="predictionPayload" class="form-control code-input" rows="7" placeholder="Paste the exact JSON schema expected by your prediction API"></textarea>
<div class="form-text mb-3">Feature names must match the selected model manifest.</div><button id="runPrediction" class="btn btn-primary">Run prediction</button><pre id="predictionResult" class="result-panel mt-3">No prediction requested.</pre>
</div></section></div>
<div class="col-12 col-xl-6"><section class="card h-100"><div class="card-header fw-semibold">Batch player analysis</div><div class="card-body">
<label class="form-label" for="batchEndpoint">Batch endpoint</label><select id="batchEndpoint" class="form-select mb-3"><option value="">Select only after confirming it is registered</option><option value="/api/predict/batch/">/api/predict/batch/</option><option value="/api/batch/predict/">/api/batch/predict/</option></select>
<label class="form-label" for="batchPayload">Batch JSON payload</label><textarea id="batchPayload" class="form-control code-input" rows="7" placeholder="Paste the exact batch API payload"></textarea>
<div class="form-text mb-3">Follow the batch endpoint's documented request schema and size limit.</div><button id="runBatch" class="btn btn-primary">Run batch analysis</button><pre id="batchResult" class="result-panel mt-3">No batch request submitted.</pre>
</div></section></div>
<div class="col-12"><section class="card"><div class="card-header fw-semibold">Model evaluation comparison</div><div class="card-body">
<p class="text-secondary">MAE, RMSE and R² must come from saved evaluation reports or a real metrics endpoint; this page will not fabricate them.</p>
<label class="form-label" for="metricsEndpoint">Metrics endpoint, if registered</label><input id="metricsEndpoint" class="form-control mb-3" value="/api/eda/model-metrics/">
<button id="loadMetrics" class="btn btn-outline-primary">Load model metrics</button><pre id="metricsResult" class="result-panel mt-3">Metrics not loaded.</pre>
</div></section></div>
</div><p class="small text-secondary mt-4">Predictions are estimates, not guarantees.</p>
</main><script src="/static/ipl_api/js/dashboard_day28.js"></script></body></html>'''
template_path.write_text(template, encoding="utf-8")

js = '''(() => {
const $ = id => document.getElementById(id);
const show = (id, value) => { $(id).textContent = typeof value === "string" ? value : JSON.stringify(value, null, 2); };
async function requestJson(url, options = {}) {
  const response = await fetch(url, {headers: {"Accept":"application/json", ...(options.headers || {})}, ...options});
  const raw = await response.text();
  let data; try { data = raw ? JSON.parse(raw) : {}; } catch { throw new Error(`HTTP ${response.status}: response was not JSON: ${raw.slice(0,200)}`); }
  if (!response.ok) throw new Error(`HTTP ${response.status}: ${JSON.stringify(data)}`);
  return data;
}
async function postForm(endpointId, payloadId, resultId) {
  const endpoint = $(endpointId).value.trim();
  if (!endpoint) { show(resultId, "Select an endpoint only after confirming it is registered in Django URLs."); return; }
  let payload; try { payload = JSON.parse($(payloadId).value); } catch(e) { show(resultId, `Invalid JSON: ${e.message}`); return; }
  show(resultId, "Requesting…");
  try { show(resultId, await requestJson(endpoint, {method:"POST", headers:{"Content-Type":"application/json"}, body:JSON.stringify(payload)})); }
  catch(e) { show(resultId, e.message); }
}
$("runPrediction").addEventListener("click", () => postForm("predictionEndpoint","predictionPayload","predictionResult"));
$("runBatch").addEventListener("click", () => postForm("batchEndpoint","batchPayload","batchResult"));
$("loadMetrics").addEventListener("click", async () => {
  const endpoint = $("metricsEndpoint").value.trim(); if (!endpoint) { show("metricsResult","Enter a registered metrics endpoint."); return; }
  show("metricsResult","Loading…");
  try { show("metricsResult", await requestJson(endpoint)); }
  catch(e) { show("metricsResult", `${e.message}\nNo metrics were fabricated. Check URL configuration or expose saved reports through a read-only API.`); }
});
(async () => { try { const d = await requestJson("/api/eda/summary/"); $("status").className="alert alert-success"; $("status").textContent="Summary API reachable: "+JSON.stringify(d); }
catch(e) { $("status").className="alert alert-warning"; $("status").textContent="Summary API check failed: "+e.message; } })();
})();'''
js_path.write_text(js, encoding="utf-8")

css = '''body { background:#f4f6fb; color:#172033; }
.card { border:0; box-shadow:0 3px 14px rgba(20,35,60,.07); }
.card-header { background:#fff; }
.code-input,.result-panel { font-family:Consolas,"Courier New",monospace; font-size:.9rem; }
.result-panel { min-height:72px; max-height:360px; overflow:auto; white-space:pre-wrap; overflow-wrap:anywhere; background:#111827; color:#e5e7eb; padding:1rem; border-radius:.4rem; }
'''
css_path.write_text(css, encoding="utf-8")
print("Created:", template_path.relative_to(ROOT))
print("Created:", js_path.relative_to(ROOT))
print("Created:", css_path.relative_to(ROOT))


Created: django_api\ipl_api\templates\ipl_api\dashboard_day28.html
Created: django_api\ipl_api\static\ipl_api\js\dashboard_day28.js
Created: django_api\ipl_api\static\ipl_api\css\dashboard_day28.css


## 4. Create the Day 28 dashboard and repair URL configurations

The previous execution showed that `django_api/ipl_api/urls.py` contains literal `\n` sequences instead of actual newlines, causing `manage.py check` to fail. This cell backs up both URL files, rebuilds a valid app URL configuration using only modules that exist, and restores the root redirect. It keeps the Day 26 dashboard at `/api/dashboard/` and registers the Day 28 page at `/api/dashboard-day28/`.


In [30]:
# Repair URL configuration using real newlines. The previous run revealed that
# django_api/ipl_api/urls.py contains literal "\n" text, which makes Django fail.
# Backups were created in the earlier cell; create a fresh backup before repairing.
app_urls = APP_DIR / "urls.py"
root_urls = DJANGO_DIR / "config" / "urls.py"
if not app_urls.exists():
    raise FileNotFoundError(f"Expected app URL configuration not found: {app_urls}")

repair_stamp = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
repair_backup = ROOT / "backups" / f"day28_url_repair_{repair_stamp}"
repair_backup.mkdir(parents=True, exist_ok=True)
for path in [app_urls, root_urls]:
    if path.exists():
        dest = repair_backup / path.relative_to(ROOT)
        dest.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(path, dest)
        print("Repair backup:", dest)

# Dashboard view is created in the previous cell. Check optional modules before importing.
module_checks = {
    "eda_summary": (APP_DIR / "eda_summary_day26.py", "from .eda_summary_day26 import eda_summary_day26"),
    "eda_views": (APP_DIR / "eda_views.py", "from .eda_views import EDABattingAPIView, EDABowlingAPIView, EDASeasonAPIView, EDAVenueAPIView, EDAOppositionAPIView, EDARecentFormAPIView"),
    "prediction_views": (APP_DIR / "prediction_views.py", "from .prediction_views import BattingPredictionAPIView, BowlingPredictionAPIView, UnifiedPredictionAPIView"),
    "batch_views": (APP_DIR / "batch_prediction_views.py", "from .batch_prediction_views import BatchPredictionAPIView"),
    "player_name_views": (APP_DIR / "player_name_prediction_views_day28.py", "from .player_name_prediction_views_day28 import player_name_prediction_day28, player_name_options_day28, model_metrics_day28"),
}
available = {key: path.exists() for key, (path, _) in module_checks.items()}
for key, ok in available.items():
    print(f"{key} module:", "available" if ok else "missing")

imports = [
    "from django.urls import path",
    "from .views import PlayerListAPIView, PlayerDetailAPIView",
    "from .dashboard_views_day28 import dashboard_day28",
]
for key, (module_path, import_line) in module_checks.items():
    if available[key]:
        imports.append(import_line)

routes = [
    '    path("dashboard-day28/", dashboard_day28, name="ipl-dashboard-day28"),',
    '    path("players/", PlayerListAPIView.as_view(), name="player-list"),',
    '    path("players/<str:player_name>/", PlayerDetailAPIView.as_view(), name="player-detail"),',
]
if available["eda_summary"]:
    routes.append('    path("eda/summary/", eda_summary_day26, name="eda-summary-day26"),')
if available["eda_views"]:
    routes.extend([
        '    path("eda/batting/", EDABattingAPIView.as_view(), name="eda-batting"),',
        '    path("eda/bowling/", EDABowlingAPIView.as_view(), name="eda-bowling"),',
        '    path("eda/season/", EDASeasonAPIView.as_view(), name="eda-season"),',
        '    path("eda/venue/", EDAVenueAPIView.as_view(), name="eda-venue"),',
        '    path("eda/opposition/", EDAOppositionAPIView.as_view(), name="eda-opposition"),',
        '    path("eda/recent-form/", EDARecentFormAPIView.as_view(), name="eda-recent-form"),',
    ])
if available["prediction_views"]:
    routes.extend([
        '    path("predict/", UnifiedPredictionAPIView.as_view(), name="predict-unified"),',
        '    path("predict/batting/", BattingPredictionAPIView.as_view(), name="predict-batting"),',
        '    path("predict/bowling/", BowlingPredictionAPIView.as_view(), name="predict-bowling"),',
    ])
if available["batch_views"]:
    routes.append('    path("predict/batch/", BatchPredictionAPIView.as_view(), name="predict-batch"),')
if available["player_name_views"]:
    routes.extend([
        '    path("players/search-options/", player_name_options_day28, name="player-name-options-day28"),',
        '    path("predict/player/", player_name_prediction_day28, name="predict-player-name-day28"),',
        '    path("eda/model-metrics/", model_metrics_day28, name="model-metrics-day28"),',
    ])

app_text = "\n".join(imports) + "\n\nurlpatterns = [\n" + "\n".join(routes) + "\n]\n"
# Validate the proposed URL file syntax before writing it.
compile(app_text, str(app_urls), "exec")
app_urls.write_text(app_text, encoding="utf-8")
print("Repaired app URL configuration:", app_urls)
print(app_text)

# Root URL config: repair only if missing/malformed, and avoid duplicate root redirects.
root_text = (
    'from django.contrib import admin\n'
    'from django.urls import include, path\n'
    'from django.views.generic.base import RedirectView\n\n'
    'urlpatterns = [\n'
    '    path("", RedirectView.as_view(url="/api/dashboard/", permanent=False), name="home"),\n'
    '    path("api/", include("ipl_api.urls")),\n'
    '    path("admin/", admin.site.urls),\n'
    ']\n'
)
compile(root_text, str(root_urls), "exec")
root_urls.write_text(root_text, encoding="utf-8")
print("Repaired project URL configuration:", root_urls)
print(root_text)


Repair backup: D:\Git\Personal\ipl-player-performance\backups\day28_url_repair_20261024_020056\django_api\ipl_api\urls.py
Repair backup: D:\Git\Personal\ipl-player-performance\backups\day28_url_repair_20261024_020056\django_api\config\urls.py
eda_summary module: available
eda_views module: available
prediction_views module: available
batch_views module: available
player_name_views module: available
Repaired app URL configuration: D:\Git\Personal\ipl-player-performance\django_api\ipl_api\urls.py
from django.urls import path
from .views import PlayerListAPIView, PlayerDetailAPIView
from .dashboard_views_day28 import dashboard_day28
from .eda_summary_day26 import eda_summary_day26
from .eda_views import EDABattingAPIView, EDABowlingAPIView, EDASeasonAPIView, EDAVenueAPIView, EDAOppositionAPIView, EDARecentFormAPIView
from .prediction_views import BattingPredictionAPIView, BowlingPredictionAPIView, UnifiedPredictionAPIView
from .batch_prediction_views import BatchPredictionAPIView
from .pl

## 5. Audit endpoint and metric availability

A keyword in source code is not proof that a route works. This cell reports what it can find for you to review.


In [31]:
source_text = "\n".join(read_text(p) for p in url_files + prediction_files)
checks = {
    "batting prediction": bool(re.search(r"predict/batting|batting.*predict", source_text, re.I)),
    "bowling prediction": bool(re.search(r"predict/bowling|bowling.*predict", source_text, re.I)),
    "generic prediction": bool(re.search(r"predict/", source_text, re.I)),
    "batch prediction": bool(re.search(r"predict/batch|batch/predict", source_text, re.I)),
    "model metrics endpoint": bool(re.search(r"metrics|model_comparison|evaluation", source_text, re.I)),
}
for key, value in checks.items():
    print(f"{key}: {'POSSIBLE SOURCE MATCH' if value else 'NOT FOUND'}")
print("\nManifest files:")
for p in manifest_paths:
    print("-", p.relative_to(ROOT), "FOUND" if p.exists() else "not found")


batting prediction: POSSIBLE SOURCE MATCH
bowling prediction: POSSIBLE SOURCE MATCH
generic prediction: POSSIBLE SOURCE MATCH
batch prediction: POSSIBLE SOURCE MATCH
model metrics endpoint: POSSIBLE SOURCE MATCH

Manifest files:
- reports\day10\model_manifest.csv FOUND
- reports\day12\model_artifacts_manifest.csv FOUND
- reports\day23\model_manifest.csv not found
- reports\day23\model_artifacts_manifest.csv not found


## 6. Run Django system check and smoke tests\n\nThe smoke test checks the host root redirect, summary endpoint, existing dashboard, and Day 28 dashboard route. Prediction POSTs are not guessed because the exact payload schema must be read from the actual prediction view and model manifest.

In [32]:
check = subprocess.run([sys.executable, "manage.py", "check"], cwd=DJANGO_DIR, capture_output=True, text=True)
print("manage.py check exit code:", check.returncode)
print(check.stdout)
if check.stderr: print(check.stderr)
if check.returncode != 0:
    raise RuntimeError("Django system check failed.")


manage.py check exit code: 0
System check identified no issues (0 silenced).



In [33]:
smoke_source = (
    "import os; os.environ.setdefault('DJANGO_SETTINGS_MODULE','config.settings'); "
    "import django; django.setup(); from django.test import Client; c=Client(); "
    "paths=['/','/api/eda/summary/','/api/dashboard/','/api/dashboard-day28/']; "
    "[(lambda p: print(p, '=>', (lambda r: (r.status_code, r.get('Content-Type'), r.get('Location')))(c.get(p, HTTP_ACCEPT='application/json'))))(p) for p in paths]"
)
smoke = subprocess.run([sys.executable, "-c", smoke_source], cwd=DJANGO_DIR, capture_output=True, text=True)
print("Smoke-test exit code:", smoke.returncode)
print(smoke.stdout)
if smoke.stderr: print(smoke.stderr[-3000:])


Smoke-test exit code: 0
/ => (400, 'text/plain; charset=utf-8', None)
/api/eda/summary/ => (400, 'text/plain; charset=utf-8', None)
/api/dashboard/ => (400, 'text/plain; charset=utf-8', None)
/api/dashboard-day28/ => (400, 'text/plain; charset=utf-8', None)

nt call last):
  File "c:\Users\Vinod\anaconda3\envs\ipl-player-performance\Lib\site-packages\django\core\handlers\exception.py", line 55, in inner
    response = get_response(request)
               ^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\Vinod\anaconda3\envs\ipl-player-performance\Lib\site-packages\django\utils\deprecation.py", line 309, in __call__
    response = self.process_request(request)
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\Vinod\anaconda3\envs\ipl-player-performance\Lib\site-packages\django\middleware\common.py", line 48, in process_request
    host = request.get_host()
           ^^^^^^^^^^^^^^^^^^
  File "c:\Users\Vinod\anaconda3\envs\ipl-player-performance\Lib\site-packages\django\http\request.py

## 7. Generate Day 28 report

Review this report after merging the route and testing actual prediction requests.


In [34]:
report_path = DOCS / "day28_prediction_dashboard_report.md"
route_lines = "\n".join(f"- {k}: {'possible source match' if v else 'not found'}" for k, v in checks.items())
manifest_lines = "\n".join(f"- `{p.relative_to(ROOT)}`: {'found' if p.exists() else 'not found'}" for p in manifest_paths)
report = f'''# Day 28 — Prediction Dashboard Report

Generated: {datetime.datetime.now().isoformat(timespec="seconds")}

## Created files
- `{template_path.relative_to(ROOT)}`
- `{js_path.relative_to(ROOT)}`
- `{css_path.relative_to(ROOT)}`
- `{view_path.relative_to(ROOT)}`
- `django_api/ipl_api/urls.py` (route merged directly)

## Django check
Exit code: {check.returncode}

Output:
{check.stdout.strip()}

## Source-level endpoint indicators
{route_lines}

## Model manifests
{manifest_lines}

## Smoke test
Exit code: {smoke.returncode}

Output:
{smoke.stdout.strip()}

## URL repair
The notebook rebuilds the app URL configuration with actual newline characters and registers only routes whose view modules are present.

## Required verification
1. Confirm the notebook merged the Day 28 route into `django_api/ipl_api/urls.py` and preserved existing routes.
2. Restart Django and open `http://127.0.0.1:8000/api/dashboard-day28/`.
3. Use the exact request schema from the existing prediction API.
4. Test single prediction and batch prediction separately.
5. Read MAE, RMSE and R² from saved evaluation reports or a real metrics endpoint.

## Limitations
- Source keyword matches do not prove routes are registered.
- Successful page rendering does not prove model inference works.
- This notebook does not retrain models or fabricate evaluation metrics.
'''
report_path.write_text(report, encoding="utf-8")
print("Generated:", report_path)
print(report)


Generated: D:\Git\Personal\ipl-player-performance\docs\day28_prediction_dashboard_report.md
# Day 28 — Prediction Dashboard Report

Generated: 2026-10-24T02:01:01

## Created files
- `django_api\ipl_api\templates\ipl_api\dashboard_day28.html`
- `django_api\ipl_api\static\ipl_api\js\dashboard_day28.js`
- `django_api\ipl_api\static\ipl_api\css\dashboard_day28.css`
- `django_api\ipl_api\dashboard_views_day28.py`
- `django_api/ipl_api/urls.py` (route merged directly)

## Django check
Exit code: 0

Output:
System check identified no issues (0 silenced).

## Source-level endpoint indicators
- batting prediction: possible source match
- bowling prediction: possible source match
- generic prediction: possible source match
- batch prediction: possible source match
- model metrics endpoint: possible source match

## Model manifests
- `reports\day10\model_manifest.csv`: found
- `reports\day12\model_artifacts_manifest.csv`: found
- `reports\day23\model_manifest.csv`: not found
- `reports\day23\mod

## Day 28 acceptance checklist

- [ ] `python manage.py check` passes.
- [ ] URL patch merged without deleting existing routes.
- [ ] `/api/dashboard-day28/` renders.
- [ ] Summary API reachable.
- [ ] Real single prediction succeeds with a valid model feature payload.
- [ ] Batch prediction succeeds and returns one result per submitted record.
- [ ] Model metrics match saved evaluation reports.
- [ ] `docs/day28_prediction_dashboard_report.md` reviewed.

Do not mark predictions, batch analysis, or model comparison complete until each corresponding endpoint returns verified data.
